In [ ]:
!pip install -U transformers accelerate sentencepiece protobuf nltk
!pip install qwen-vl-utils

In [ ]:
from pathlib import Path

import torch
import torch.nn.functional as F
from IPython.display import FileLink, display
from tqdm.auto import tqdm
from transformers import AutoProcessor, LlavaForConditionalGeneration


VLM_CHOICE = "llava"
VLM_REGISTRY = {
    "llava": {
        "model_id": "llava-hf/llava-1.5-7b-hf",
        "representation": "contextual_hidden_states_of_exact_target_prompt",
    }
}
vlm_cfg = VLM_REGISTRY[VLM_CHOICE]


# Original SANA input discovery:
# def find_input(name):
#     return next(Path("/kaggle/input").rglob(name))
# sana = torch.load(find_input("geneval_vocab1080_native_exact.pt"), ...)
# eval_pairs = torch.load(find_input("sana_llava_geneval_exact_pairs.pt"), ...)

SD3_TARGET_PATH = Path(
    "/kaggle/input/datasets/aveenhussein/classes-failed-me-and-i-am-sad/"
    "sd3_t5_geneval_vocab1080_compact.pt"
)
sd3 = torch.load(
    SD3_TARGET_PATH,
    map_location="cpu",
    weights_only=False,
)

texts = list(sd3["conditioned_texts"])
target_mask = sd3["attention_mask"].bool()
target_ids = sd3["input_ids"]
target_special = sd3["special_tokens_mask"].bool()
target_offsets = sd3["offset_mapping"]
target_active = sd3["Y_pred_active"]
row_offsets = torch.cat([
    torch.zeros(1, dtype=torch.long),
    target_mask.sum(1).cumsum(0),
])

EVAL_PROMPTS = int(sd3["evaluation_prompt_count"])
eval_row_count = int(target_mask[:EVAL_PROMPTS].sum())
eval_pairs = {
    "Y": target_active[:eval_row_count].contiguous(),
    "attention_mask": target_mask[:EVAL_PROMPTS].clone(),
    "target_input_ids": target_ids[:EVAL_PROMPTS].clone(),
    "prompt_texts": list(sd3["prompt_texts"][:EVAL_PROMPTS]),
    "conditioned_texts": list(sd3["conditioned_texts"][:EVAL_PROMPTS]),
    "ids": list(sd3["ids"][:EVAL_PROMPTS]),
    "chi_prompt": sd3["chi_prompt"],
    "source_model": vlm_cfg["model_id"],
    "target_model": sd3["text_encoder_model_id"],
}

model = LlavaForConditionalGeneration.from_pretrained(
    vlm_cfg["model_id"],
    torch_dtype=torch.float16,
    device_map="auto",
).eval()
tokenizer = AutoProcessor.from_pretrained(vlm_cfg["model_id"]).tokenizer
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

language_model = (
    model.language_model
    if hasattr(model, "language_model")
    else model.model.language_model
)
source_backbone = (
    language_model.model
    if hasattr(language_model, "model")
    else language_model
)
source_device = model.get_input_embeddings().weight.device

print("VLM:", vlm_cfg["model_id"])
print("Target:", sd3["text_encoder_model_id"])
print("Fitting prompts:", len(texts))
print("Active target rows:", len(target_active))


In [ ]:
import json
from collections import defaultdict
# from transformers import AutoModelForCausalLM  # Original SD3 prior loader.

SEAM_K = 4
SEAM_SIGMA = 2.0
SEAM_LAMBDA = 1e-2  # Numerical regularization after normalizing Ridge inputs.
# EVAL_PROMPTS = 553  # Original fixed SD3 value.
EVAL_PROMPTS = int(sd3["evaluation_prompt_count"])
fit_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seam_features(sequence):
    """DFT along token positions; keep K low-frequency bins and their phase."""
    sequence = sequence.float()
    spectrum = torch.fft.rfft(sequence, dim=0)
    bins = min(SEAM_K, spectrum.shape[0])
    k = torch.arange(bins, device=sequence.device, dtype=torch.float32)
    spectrum = spectrum[:bins] * torch.exp(-k.square() / (2 * SEAM_SIGMA**2))[:, None]
    packed = torch.zeros(SEAM_K, 2, sequence.shape[1], device=sequence.device)
    packed[:bins, 0], packed[:bins, 1] = spectrum.real, spectrum.imag
    return packed.flatten()


def inverse_seam(features, length, prior=None):
    """Undo the Gaussian spectral filter before inverse DFT; keep prior's high bins."""
    d = features.numel() // (2 * SEAM_K)
    parts = features.float().reshape(SEAM_K, 2, d)
    values = torch.complex(parts[:, 0], parts[:, 1])
    count = length // 2 + 1
    bins = min(SEAM_K, count)
    k = torch.arange(bins, device=values.device, dtype=torch.float32)
    gain = torch.exp(-k.square() / (2 * SEAM_SIGMA**2))
    spectrum = (torch.fft.rfft(prior.float(), dim=0) if prior is not None
                else torch.zeros(count, d, dtype=values.dtype, device=values.device))
    spectrum[:bins] += values[:bins] / gain[:, None]
    return torch.fft.irfft(spectrum, n=length, dim=0)


def fit_seam_ridge(X, Y):
    """Numerically stable closed-form Ridge, no SGD; works for wide matrices."""
    X = X.double().to(fit_device)
    Y = Y.float().cpu()
    x_mean = X.mean(0, keepdim=True)
    x_std = (X - x_mean).square().mean(0, keepdim=True).sqrt().clamp_min(1e-5)
    X = (X - x_mean) / x_std
    x_scale = X.norm(dim=1).mean().clamp_min(1e-8)
    X = X / x_scale
    y_mean = Y.mean(0, keepdim=True)

    dual = X.shape[0] <= X.shape[1]
    gram = X @ X.T if dual else X.T @ X
    gram.diagonal().add_(SEAM_LAMBDA)
    cholesky = torch.linalg.cholesky(gram)
    weights = []
    for start in range(0, Y.shape[1], 256):
        target = (Y[:, start:start+256] - y_mean[:, start:start+256]).double().to(fit_device)
        rhs = target if dual else X.T @ target
        weights.append(torch.cholesky_solve(rhs, cholesky).float())
    return dict(x_mean=x_mean.float(), x_std=x_std.float(), x_scale=float(x_scale),
                train=X.float() if dual else None, weights=torch.cat(weights, dim=1),
                y_mean=y_mean.to(fit_device), dual=dual)


@torch.inference_mode()
def apply_seam_ridge(state, X, batch_size=256):
    output = []
    for i in range(0, len(X), batch_size):
        x = X[i:i+batch_size].float().to(fit_device)
        x = ((x - state["x_mean"]) / state["x_std"]) / state["x_scale"]
        if state["dual"]:
            x = x @ state["train"].T
        output.append((x @ state["weights"] + state["y_mean"]).cpu())
    return torch.cat(output)


def embedding_metrics(prediction, target):
    p, t = prediction.float(), target.float()
    return {"cosine": F.cosine_similarity(p, t, dim=1).mean().item(),
            "mse": F.mse_loss(p, t).item(),
            "norm_mae": (p.norm(dim=1)-t.norm(dim=1)).abs().mean().item()}


# ============================================================
# EXTRACT ALIGNED LLAVA / T5 EMBEDDINGS
# ============================================================

# Original SD3 code loaded the complete T5 model here.
# SD3's compact server artifact already contains the required T5 lookup priors.
# ============================================================

lookup_token_ids = sd3["lookup_token_ids"].long()
lookup_embeddings = sd3["lookup_embeddings"].half()


# ============================================================
# BUILD ORDERED TOKEN GROUPS
# ============================================================

spectral_rows = []
merge_target_rows = []
sd3_target_rows = []

prompt_rows = []
position_rows = []
token_rows = []

reverse_groups = []


@torch.inference_mode()
def extract_prompt(prompt_index):
    encoded = tokenizer(
        texts[prompt_index],
        add_special_tokens=True,
        return_offsets_mapping=True,
        return_special_tokens_mask=True,
        return_tensors="pt",
    )

    source_ids = encoded[
        "input_ids"
    ][0]

    source_offsets = encoded.pop(
        "offset_mapping"
    )[0]

    source_special = encoded.pop(
        "special_tokens_mask"
    )[0].bool()

    source_attention = encoded[
        "attention_mask"
    ][0].bool()

    source_hidden = source_backbone(
        input_ids=encoded[
            "input_ids"
        ].to(source_device),
        attention_mask=encoded[
            "attention_mask"
        ].to(source_device),
        use_cache=False,
        return_dict=True,
    )[0][0]

    positions = torch.where(
        target_mask[prompt_index]
    )[0]

    token_ids = target_ids[
        prompt_index,
        positions,
    ]

    target_is_special = target_special[
        prompt_index,
        positions,
    ]

    start = int(
        row_offsets[prompt_index]
    )

    end = int(
        row_offsets[prompt_index + 1]
    )

    contextual_targets = (
        target_active[start:end]
    )

    prior_rows = torch.searchsorted(
        lookup_token_ids,
        token_ids.long(),
    )
    token_priors = lookup_embeddings[
        prior_rows
    ]

    source_regular = torch.where(
        source_attention
        & ~source_special
        & (
            source_offsets[:, 1]
            > source_offsets[:, 0]
        )
    )[0]

    source_special_positions = (
        torch.where(
            source_attention
            & source_special
        )[0]
    )

    source_sequences = [
        None
    ] * len(positions)

    assigned_source = [
        None
    ] * len(positions)

    regular_targets = torch.where(
        ~target_is_special
    )[0]

    if len(regular_targets):
        target_spans = target_offsets[
            prompt_index,
            positions[regular_targets],
        ]

        source_spans = source_offsets[
            source_regular
        ]

        overlap = (
            torch.minimum(
                target_spans[:, None, 1],
                source_spans[None, :, 1],
            )
            - torch.maximum(
                target_spans[:, None, 0],
                source_spans[None, :, 0],
            )
        ).clamp_min(0)

        target_centers = (
            target_spans.float().mean(1)
        )

        source_centers = (
            source_spans.float().mean(1)
        )

        for row, target_local in enumerate(
            regular_targets.tolist()
        ):
            matches = torch.where(
                overlap[row] > 0
            )[0]

            if len(matches) == 0:
                nearest = (
                    target_centers[row]
                    - source_centers
                ).abs().argmin()

                matches = nearest.view(1)

            ordered_source = (
                source_regular[matches]
            )

            source_sequences[
                target_local
            ] = ordered_source

            if float(
                overlap[
                    row,
                    matches,
                ].max()
            ) > 0:
                best = matches[
                    overlap[
                        row,
                        matches,
                    ].argmax()
                ]
            else:
                best = matches[0]

            assigned_source[
                target_local
            ] = int(
                source_regular[best]
            )

    special_targets = torch.where(
        target_is_special
    )[0]

    for order, target_local in enumerate(
        special_targets.tolist()
    ):
        source_position = int(
            source_special_positions[
                min(
                    order,
                    len(
                        source_special_positions
                    ) - 1,
                )
            ]
        )

        source_sequences[
            target_local
        ] = torch.tensor([
            source_position
        ])

        assigned_source[
            target_local
        ] = source_position

    for local_index, source_indices in enumerate(
        source_sequences
    ):
        sequence = source_hidden[
            source_indices.to(source_hidden.device)
        ]

        # SEAM input: ordered LLaVA sequence.
        spectral_rows.append(
            seam_features(
                sequence
            ).cpu().half()
        )

        # First Ridge target: one LLaVA-space embedding.
        # The final contextual token represents the
        # complete ordered subtoken sequence.
        merge_target_rows.append(
            sequence[-1].cpu().half()
        )

        # Second KRR target: the contextual T5
        # representation SD3 actually consumes.
        sd3_target_rows.append(
            contextual_targets[
                local_index
            ]
        )

        prompt_rows.append(
            prompt_index
        )

        position_rows.append(
            int(
                positions[local_index]
            )
        )

        token_rows.append(
            int(
                token_ids[local_index]
            )
        )

    grouped_targets = defaultdict(list)

    for target_local, source_index in enumerate(
        assigned_source
    ):
        grouped_targets[
            source_index
        ].append(target_local)

    for source_index in sorted(
        grouped_targets
    ):
        local_rows = grouped_targets[
            source_index
        ]

        reverse_groups.append({
            "source": (
                source_hidden[
                    source_index
                ].cpu().half()
            ),
            "target": (
                contextual_targets[
                    local_rows
                ]
            ),
            "prior": (
                token_priors[
                    local_rows
                ]
            ),
            "target_ids": (
                token_ids[
                    local_rows
                ]
            ),
            "rows": torch.tensor([
                start + row
                for row in local_rows
            ]),
            "key": (
                int(
                    source_ids[
                        source_index
                    ]
                ),
                tuple(
                    int(
                        token_ids[row]
                    )
                    for row in local_rows
                ),
            ),
            "is_eval": (
                prompt_index
                < EVAL_PROMPTS
            ),
        })


for prompt_index in tqdm(
    range(len(texts)),
    desc="Building SEAM alignments",
):
    extract_prompt(prompt_index)


spectral_rows = torch.stack(
    spectral_rows
)

merge_target_rows = torch.stack(
    merge_target_rows
)

all_sd3_targets = torch.stack(
    sd3_target_rows
)

all_prompt_indices = torch.tensor(
    prompt_rows,
    dtype=torch.int32,
)

all_target_positions = torch.tensor(
    position_rows,
    dtype=torch.int16,
)

all_token_ids = torch.tensor(
    token_rows,
    dtype=torch.int32,
)


# ============================================================


# All embeddings and token priors have been extracted: free GPU memory before Ridge.
del model, language_model, source_backbone, lookup_token_ids, lookup_embeddings
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ONE CALIBRATION ROW PER UNIQUE T5 TOKEN ID
# ============================================================

first_row_by_token = {}

for row_index, token_id in enumerate(
    all_token_ids.tolist()
):
    if token_id not in first_row_by_token:
        first_row_by_token[
            token_id
        ] = row_index

training_rows = torch.tensor(
    list(
        first_row_by_token.values()
    ),
    dtype=torch.long,
)


# ============================================================
# FIRST KRR: ORIGINAL SEAM MANY-TO-ONE ALIGNMENT
# ============================================================

seam_state = fit_seam_ridge(
    spectral_rows[
        training_rows
    ],
    merge_target_rows[
        training_rows
    ],
)

merged_llava = apply_seam_ridge(
    seam_state,
    spectral_rows,
)

mean_llava_norm = (
    merge_target_rows[
        training_rows
    ].float().norm(
        dim=1
    ).mean()
)

merged_llava = (
    F.normalize(
        merged_llava,
        dim=1,
    )
    * mean_llava_norm
)


eval_row_count = int(
    target_mask[
        :EVAL_PROMPTS
    ].sum()
)

seam_alignment_metrics = (
    embedding_metrics(
        merged_llava[
            :eval_row_count
        ],
        merge_target_rows[
            :eval_row_count
        ],
    )
)


# These are consumed by the existing Cholesky + polynomial
# KRR cells that follow this cell.

X_cpu = (
    merged_llava[
        training_rows
    ].half().contiguous()
)

Y_cpu = (
    all_sd3_targets[
        training_rows
    ].contiguous()
)

fit_prompt_indices = (
    all_prompt_indices[
        training_rows
    ]
)

fit_positions = (
    all_target_positions[
        training_rows
    ]
)

fit_token_ids = (
    all_token_ids[
        training_rows
    ]
)


# Replace the old averaged LLaVA evaluation rows with
# the order-aware SEAM-merged LLaVA rows.

eval_pairs = dict(
    eval_pairs
)

eval_pairs["X"] = (
    merged_llava[
        :eval_row_count
    ].half().contiguous()
)


# ============================================================


# ============================================================
# REVERSE SEAM: PREDICT TOKEN SEQUENCE SPECTRUM FROM ONE VECTOR
# All prompts can contribute to the fit, including the 553 exported prompts.
# ============================================================

# One example per source token + ordered target token tuple; no evaluation holdout.
reverse_train = list({g["key"]: g for g in reversed(reverse_groups)}.values())
reverse_eval = [g for g in reverse_groups if g["is_eval"]]


def reverse_input(group):
    # Supply the known target sequence length explicitly to Ridge.
    return torch.cat([group["source"].float(), torch.tensor([len(group["target"])])])


reverse_X = torch.stack([reverse_input(g) for g in reverse_train])
reverse_Y = torch.stack([
    torch.cat([seam_features(g["target"]),
               seam_features(g["target"].float() - g["prior"].float())])
    for g in reverse_train
])
reverse_state = fit_seam_ridge(reverse_X, reverse_Y)

predicted = apply_seam_ridge(
    reverse_state, torch.stack([reverse_input(g) for g in reverse_eval]))
spectral_size = reverse_Y.shape[1] // 2
count = int(target_mask[:EVAL_PROMPTS].sum())
dim = all_sd3_targets.shape[1]
plain_reverse_prediction = torch.zeros(count, dim)
informed_reverse_prediction = torch.zeros(count, dim)

for i, group in enumerate(reverse_eval):
    length = len(group["target"])
    rows = group["rows"]
    plain_reverse_prediction[rows] = inverse_seam(predicted[i, :spectral_size], length)
    informed_reverse_prediction[rows] = inverse_seam(
        predicted[i, spectral_size:], length, prior=group["prior"])

metrics = {
    "forward_seam_alignment": seam_alignment_metrics,
    "reverse_seam": embedding_metrics(plain_reverse_prediction, all_sd3_targets[:count]),
    "token_informed_reverse_seam": embedding_metrics(informed_reverse_prediction, all_sd3_targets[:count]),
}
print(json.dumps(metrics, indent=2))


# ============================================================
# EXPORT RECONSTRUCTED SD3 ACTIVE T5 CONDITIONING
# ============================================================

def save_reverse(name, prediction):
    path = Path(f"/kaggle/working/sd3_{name}_projected_compact.pt")
    torch.save({
        "format_version": 2,
        "kind": "sd3_projected_official_geneval_native_sequence",
        "projected_text_mode": "replace_content_and_eos_keep_native_padding",
        "Y_pred_active": prediction.to(torch.float16).contiguous(),
        "conditioning_storage": "active_tokens_only",
        "dense_shape": [EVAL_PROMPTS, target_mask.shape[1], prediction.shape[1]],
        "attention_mask": sd3["attention_mask"][:EVAL_PROMPTS].clone(),
        "model_attention_mask": sd3["model_attention_mask"][:EVAL_PROMPTS].clone(),
        "native_padding_policy": "retain_native_contextual_padding",
        "token_ids": sd3["input_ids"][:EVAL_PROMPTS].clone(),
        "prompt_texts": list(sd3["prompt_texts"][:EVAL_PROMPTS]),
        "conditioned_texts": list(sd3["conditioned_texts"][:EVAL_PROMPTS]),
        "ids": list(sd3["ids"][:EVAL_PROMPTS]),
        "chi_prompt": sd3["chi_prompt"],
        "source_model": vlm_cfg["model_id"],
        "target_model": sd3["text_encoder_model_id"],
        "projection_method": name,
        "embedding_metrics": metrics[name],
    }, path)
    display(FileLink(str(path)))


save_reverse("reverse_seam", plain_reverse_prediction)
save_reverse("token_informed_reverse_seam", informed_reverse_prediction)


In [ ]:
fit_device = torch.device("cuda")


def normalize_rows(tensor):
    return F.normalize(tensor, dim=1, eps=1e-12)


def fit_cholesky_whitener(tensor, epsilon=1e-3):
    mean = tensor.mean(0, keepdim=True)
    centered = tensor - mean
    covariance = centered.T @ centered / (len(centered) - 1)
    cholesky = torch.linalg.cholesky(
        covariance
        + epsilon * torch.eye(covariance.shape[0], device=tensor.device)
    )
    whitened = torch.linalg.solve_triangular(
        cholesky,
        centered.T,
        upper=False,
    ).T
    return whitened, mean, cholesky


def polynomial_kernel(first, second):
    return (0.5003 * (first @ second.T) + 0.2) ** 2


def projection_metrics(prediction, target, batch_size=256):
    prediction = normalize_rows(prediction)
    target = normalize_rows(target)
    correct = F.cosine_similarity(prediction, target, dim=1)
    wrong_sum = 0.0
    wrong_count = 0
    top1_correct = 0

    for start in range(0, len(prediction), batch_size):
        end = min(start + batch_size, len(prediction))
        similarities = prediction[start:end] @ target.T
        rows = torch.arange(end - start, device=prediction.device)
        columns = torch.arange(start, end, device=prediction.device)
        diagonal = similarities[rows, columns]
        wrong_sum += float(similarities.sum() - diagonal.sum())
        wrong_count += similarities.numel() - len(diagonal)
        top1_correct += int((similarities.argmax(1) == columns).sum())

    l2 = torch.norm(prediction - target, dim=1)
    metrics = {
        "correct_cosine": float(correct.mean()),
        "wrong_cosine": wrong_sum / wrong_count,
        "top1": top1_correct / len(prediction),
        "separation_gap": float(correct.mean()) - wrong_sum / wrong_count,
        "mse": float(F.mse_loss(prediction, target)),
        "mean_l2": float(l2.mean()),
        "normalized_l2_error": float((l2 / 2).mean()),
        "l2_closeness": 1 - float((l2 / 2).mean()),
        "minimum_cosine": float(correct.min()),
        "maximum_cosine": float(correct.max()),
    }
    print(metrics)
    return metrics


X_raw = X_cpu.float().to(fit_device)
Y_raw = Y_cpu.float().to(fit_device)

X_norm = normalize_rows(X_raw)
X_norm_mean = X_norm.mean(0, keepdim=True)
X_norm = normalize_rows(X_norm - X_norm_mean)

Y_runtime = normalize_rows(Y_raw)
Y_norm, Y_mean, Y_cholesky = fit_cholesky_whitener(Y_runtime)

K = polynomial_kernel(X_norm, X_norm)
K = (K + K.T) / 2
lambda_reg = 1e-5
kernel_cholesky = torch.linalg.cholesky(
    K + lambda_reg * torch.eye(len(K), device=fit_device)
)
direction_coefficients = torch.cholesky_solve(
    Y_norm,
    kernel_cholesky,
)

target_norm = Y_raw.norm(dim=1, keepdim=True).clamp_min(1e-12)
target_log_norm = target_norm.log()
log_norm_mean = target_log_norm.mean(0, keepdim=True)
log_norm_std = target_log_norm.std(0, keepdim=True).clamp_min(1e-6)
norm_coefficients = torch.cholesky_solve(
    (target_log_norm - log_norm_mean) / log_norm_std,
    kernel_cholesky,
)
norm_min = float(target_norm.min())
norm_max = float(target_norm.max())

print("Kernel:", tuple(K.shape))
print("Direction coefficients:", tuple(direction_coefficients.shape))
print("Lambda:", lambda_reg)



In [ ]:
def transform_source(source):
    return normalize_rows(normalize_rows(source) - X_norm_mean)


@torch.inference_mode()
def predict_from_kernel(kernel_rows, restore_scale):
    direction = normalize_rows(
        (kernel_rows @ direction_coefficients) @ Y_cholesky.T + Y_mean
    )
    if not restore_scale:
        return direction

    scale = (
        (kernel_rows @ norm_coefficients) * log_norm_std + log_norm_mean
    ).exp().clamp(norm_min, norm_max)
    return direction * scale


@torch.inference_mode()
def project_rows(source_rows, restore_scale, batch_size=256):
    parts = []
    for start in tqdm(
        range(0, len(source_rows), batch_size),
        desc="Projecting evaluation rows",
    ):
        source = source_rows[start : start + batch_size].float().to(fit_device)
        kernel_rows = polynomial_kernel(transform_source(source), X_norm)
        parts.append(predict_from_kernel(kernel_rows, restore_scale).cpu())
    return torch.cat(parts)


Y_train_direction = predict_from_kernel(K, restore_scale=False)
print("Training direction metrics")
train_metrics = projection_metrics(Y_train_direction, Y_runtime)

Y_eval_direction = project_rows(eval_pairs["X"], restore_scale=False)
Y_eval_native = eval_pairs["Y"].float()
print("Official evaluation direction metrics")
eval_metrics = projection_metrics(
    Y_eval_direction.to(fit_device),
    normalize_rows(Y_eval_native.to(fit_device)),
)

Y_train_projected = predict_from_kernel(K, restore_scale=True)
Y_eval_projected = project_rows(eval_pairs["X"], restore_scale=True)
train_native_norm = Y_raw.norm(dim=1)
train_projected_norm = Y_train_projected.norm(dim=1)
eval_native_norm = Y_eval_native.norm(dim=1)
eval_projected_norm = Y_eval_projected.norm(dim=1)
scale_metrics = {
    "scale_training_mae": float((
        train_projected_norm - train_native_norm
    ).abs().mean()),
    "scale_heldout_mae": float((
        eval_projected_norm - eval_native_norm
    ).abs().mean()),
    "scale_heldout_correlation": float(torch.corrcoef(torch.stack([
        eval_projected_norm,
        eval_native_norm,
    ]))[0, 1]),
}
print("Official evaluation scale metrics")
print(scale_metrics)



In [ ]:
PROJECTOR_PATH = Path(
    "/kaggle/working/sd3_llava_geneval_vocab1080_projector_token_scale.pt"
)
PROJECTED_PATH = Path(
    "/kaggle/working/"
    "sd3_llava_geneval_vocab1080_projected_compact_token_scale.pt"
)

projector = {
    "format_version": 1,
    "kind": "sd3_t5_vocab1080_token_kernel_projector",
    "source_model": vlm_cfg["model_id"],
    "target_model": sd3["text_encoder_model_id"],
    "source_representation": vlm_cfg["representation"],
    "target_representation": "native_sd3_t5_content_and_eos_sequence",
    "calculation_prompt_count": len(texts),
    "training_pair_count": len(X_cpu),
    "training_pair_policy": "first_occurrence_per_unique_target_token_id",
    "training_token_ids": fit_token_ids,
    "prompt_texts": list(sd3["prompt_texts"]),
    "ids": list(sd3["ids"]),
    "chi_prompt": sd3["chi_prompt"],
    "source_add_special_tokens": True,
    "source_tokenizer_padding_side": "right",
    "official_geneval_used_for_fitting": True,
    "prompt_indices": fit_prompt_indices,
    "target_positions": fit_positions,
    "X_norm_mean": X_norm_mean.cpu(),
    "train_features": X_norm.cpu(),
    "kernel": {
        "kind": "polynomial",
        "sigma": 0.5003,
        "degree": 2,
        "coef0": 0.2,
        "lambda_reg": lambda_reg,
    },
    "coefficients": direction_coefficients.cpu(),
    "Y_mean": Y_mean.cpu(),
    "Y_cholesky": Y_cholesky.cpu(),
    "Y_cholesky_epsilon": 1e-3,
    "norm_coefficients": norm_coefficients.cpu(),
    "Y_log_norm_mean": log_norm_mean.cpu(),
    "Y_log_norm_std": log_norm_std.cpu(),
    "Y_norm_min": norm_min,
    "Y_norm_max": norm_max,
    "training_metrics": train_metrics,
    "heldout_metrics": eval_metrics,
    **scale_metrics,
}

projected = {
    "format_version": 2,
    "kind": "sd3_projected_official_geneval_native_sequence",
    "projected_text_mode": "replace_content_and_eos_keep_native_padding",
    "Y_pred_active": Y_eval_projected.to(torch.float16).contiguous(),
    "conditioning_storage": "active_tokens_only",
    "dense_shape": [
        *eval_pairs["attention_mask"].shape,
        Y_eval_projected.shape[1],
    ],
    "attention_mask": eval_pairs["attention_mask"].clone(),
    "model_attention_mask": sd3["model_attention_mask"][:EVAL_PROMPTS].clone(),
    "special_tokens_mask": sd3["special_tokens_mask"][:EVAL_PROMPTS].clone(),
    "offset_mapping": sd3["offset_mapping"][:EVAL_PROMPTS].clone(),
    "native_padding_policy": "retain_native_contextual_padding",
    "token_ids": eval_pairs["target_input_ids"].clone(),
    "prompt_texts": list(eval_pairs["prompt_texts"]),
    "conditioned_texts": list(eval_pairs["conditioned_texts"]),
    "ids": list(eval_pairs["ids"]),
    "chi_prompt": eval_pairs["chi_prompt"],
    "source_model": eval_pairs["source_model"],
    "target_model": eval_pairs["target_model"],
    "conditioning_layout": "native_sd3_t5_positions_active_rows_only",
    "projector_kind": projector["kind"],
    "projector_training_prompt_count": len(texts),
    "projector_training_pair_count": len(X_cpu),
    "projector_training_policy": projector["training_pair_policy"],
    "projector_training_metrics": train_metrics,
    "projector_heldout_metrics": eval_metrics,
    "scale_projection": "kernel_ridge_log_l2_norm",
    **scale_metrics,
}

torch.save(projector, PROJECTOR_PATH)
torch.save(projected, PROJECTED_PATH)

print("Saved:", PROJECTOR_PATH)
print("Saved:", PROJECTED_PATH)
display(FileLink(str(PROJECTOR_PATH)))
display(FileLink(str(PROJECTED_PATH)))
